In [1]:
from sage.all import *
import random
import time

F2 = GF(2)
R.<x> = PolynomialRing(F2)

def compute_product(polynomial_set):
    if not polynomial_set:
        return R(1)
    product = R(1)
    for poly in polynomial_set:
        product *= poly
    return product

def generate_random_polynomial_set_by_degree(all_polys, target_degree, seed):
    local_random = random.Random(seed)
    shuffled_polys = list(all_polys)
    local_random.shuffle(shuffled_polys)
    selected_polys = []
    current_degree = 0
    for poly_info in shuffled_polys:
        poly = poly_info['poly']
        deg = poly.degree()
        if current_degree + deg <= target_degree:
            selected_polys.append(poly)
            current_degree += deg
    if current_degree == target_degree:
        return selected_polys
    else:
        return []

def check_linear_independence_optimized(new_poly_product, basis_vectors_span, max_degree):
    coeffs = new_poly_product.list() + [0] * (max_degree + 1 - len(new_poly_product.list()))
    v = vector(F2, coeffs)
    return v not in basis_vectors_span

def randomized_greedy_search(all_mod_polys, degree_sequence, target_sets, max_attempts_per_set=500000):
    print(f"Begin performing a randomized greedy search with the goal of {target_sets} linearly independent polynomial combinations.")
    print(f"The following degrees will be tried in sequence: {degree_sequence}")
    final_products = []
    final_selected_sets = []
    max_possible_degree = max(degree_sequence)
    basis_matrix = Matrix(F2, 0, max_possible_degree + 1)
    basis_vectors_span = basis_matrix.row_space()
    current_degree_idx = 0
    for set_idx in range(target_sets):
        print(f"\n--- Looking for the {set_idx + 1}/{target_sets}th combination ---")
        found_independent_in_any_degree = False
        for i in range(current_degree_idx, len(degree_sequence)):
            target_degree = degree_sequence[i]
            print(f"The target degree currently being attempted: {target_degree}")
            found_independent_for_current_degree = False
            for attempt in range(max_attempts_per_set):
                seed_value = int(time.time() * 1000) + int(set_idx) * int(max_attempts_per_set) + int(attempt)
                candidate_set = generate_random_polynomial_set_by_degree(
                    all_mod_polys, target_degree, seed_value
                )
                if not candidate_set:
                    continue
                candidate_product = compute_product(candidate_set)
                if candidate_product.degree() != target_degree:
                    continue
                if check_linear_independence_optimized(candidate_product, basis_vectors_span, max_possible_degree):
                    print(f"Success! A linearly independent combination was found on the {attempt + 1}th attempt.")
                    final_products.append(candidate_product)
                    final_selected_sets.append(candidate_set)
                    coeffs = candidate_product.list() + [0] * (max_possible_degree + 1 - len(candidate_product.list()))
                    v = vector(F2, coeffs)
                    basis_matrix = basis_matrix.stack(v)
                    basis_vectors_span = basis_matrix.row_space()
                    found_independent_for_current_degree = True
                    found_independent_in_any_degree = True
                    break
            if found_independent_for_current_degree:
                current_degree_idx = i
                break
        if not found_independent_in_any_degree:
            print(f"\nAfter trying all available degrees, the algorithm failed to find the {set_idx + 1}th combination. The algorithm terminated prematurely.")
            break
    print("\n=========================================================")
    print(f"Search complete. A total of {len(final_products)} linearly independent polynomial combinations were successfully found.")
    if final_products:
        degree_counts = {}
        for poly in final_products:
            deg = poly.degree()
            degree_counts[deg] = degree_counts.get(deg, 0) + 1
        print("\nThe degree distribution of the final combination:")
        for deg in sorted(degree_counts.keys(), reverse=True):
            print(f"degree{deg}: {degree_counts[deg]} ")
    return final_selected_sets, final_products

if __name__ == '__main__':
    def generate_mod_polynomials(d, count, u):
        mod_polys = []
        if d > 0:
            c = 0
            for coeffs_int in range(2**d):
                poly_coeffs = [0] * (d + 1); poly_coeffs[d] = 1
                for i in range(d): poly_coeffs[i] = (coeffs_int >> i) & 1
                poly = R(poly_coeffs)
                if poly.is_irreducible():
                    mod_polys.append(poly); c += 1
                    if c >= count: break
        mod_polys_u = []
        for i in range(min(count, len(mod_polys))):
            if isinstance(u, list) and len(u) > i: mod_polys_u.append([str(mod_polys[i]), u[i]])
            else: mod_polys_u.append([str(mod_polys[i]), u])
        return mod_polys_u

    u=[5, 3, 1, 1, 1, 1, 1, 1, 0, 1, 4]
    num=[2, 1, 2, 3, 6, 9, 17, 30, 0, 66]
    all_mod_polys_u = []
    for i in range(10):
        all_mod_polys_u += generate_mod_polynomials(i + 1, num[i], u[i])

    base_polys_for_search = []
    for p_str, max_use in all_mod_polys_u:
        poly = R(p_str)
        base_polys_for_search.append({'poly': poly})

    print(f"The base polynomial library has been generated, containing a total of {len(base_polys_for_search)} polynomials.")

    TARGET_SETS = 163
    DEGREE_SEQUENCE = [162,161, 160, 159]

    start_time = time.time()

    final_sets, final_products = randomized_greedy_search(
        all_mod_polys=base_polys_for_search,
        degree_sequence=DEGREE_SEQUENCE,
        target_sets=TARGET_SETS
    )

    end_time = time.time()
    print(f"\nTotal time elapsed: {end_time - start_time:.2f} seconds")


The base polynomial library has been generated, containing a total of 136 polynomials.
Begin performing a randomized greedy search with the goal of 163 linearly independent polynomial combinations.
The following degrees will be tried in sequence: [162, 161, 160, 159]

--- Looking for the 1/163th combination ---
The target degree currently being attempted: 162
Success! A linearly independent combination was found on the 1th attempt.

--- Looking for the 2/163th combination ---
The target degree currently being attempted: 162
Success! A linearly independent combination was found on the 1th attempt.

--- Looking for the 3/163th combination ---
The target degree currently being attempted: 162
Success! A linearly independent combination was found on the 1th attempt.

--- Looking for the 4/163th combination ---
The target degree currently being attempted: 162
Success! A linearly independent combination was found on the 1th attempt.

--- Looking for the 5/163th combination ---
The target degr

In [2]:
# print(final_products)
# def save_BD_basis_to_file(BD, filename):
#     with open(filename, "w", encoding="utf-8") as f:
#         for i, basis_element in enumerate(BD):
#             poly_str = str(basis_element)
#             poly_str = poly_str.replace("*", "")
#             f.write(poly_str + "\n")
#     print(f"BD base has been saved to {filename}")

# for i in range(570):
#     final_products.append(x^(571+i))
# save_BD_basis_to_file(final_products, "2^571_l(2D)_pn_basis.txt")